# Triton-Lab: Vector Addition

This notebook is a short experiment companion to the Python implementation in `src/triton_lab/vector_add.py`.

**Goal:** check correctness, run a custom Triton kernel, and measure it against PyTorch. This is a learning baseline; no speedup is assumed in advance.

## 1. Check the runtime

Select a GPU runtime in Colab before running the cells. The timings depend on the GPU and software versions, so record them with every experiment.

In [ ]:
import torch
import triton
from pathlib import Path
import os, sys

print('PyTorch:', torch.__version__)
print('Triton:', triton.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))

## 2. Load the project code

The actual kernel lives in a `.py` file so it can be reused by tests and the benchmark. This avoids keeping separate copies of the same implementation in several places.

In [ ]:
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'triton_lab').exists():
    raise FileNotFoundError('Run this notebook from the cloned triton-lab repository.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
from triton_lab.vector_add import vector_add, vector_add_out
print('Repository:', ROOT)

## 3. Check correctness first

The test length is 1003, not a multiple of the block size (256). This checks the **mask** in the last block: lanes beyond the end of the array must not read or write valid data. We compare the Triton result to PyTorch's result.

In [ ]:
n = 1003
x = torch.randn(n, device='cuda', dtype=torch.float32)
y = torch.randn_like(x)
actual = vector_add(x, y, block_size=256)
expected = x + y

print('All outputs close:', torch.allclose(actual, expected, rtol=1e-5, atol=1e-6))
torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)

## 4. What the kernel does

For each Triton program, `tl.program_id(0)` gives the program's block number. With a block size of 256, program 0 handles offsets 0–255, program 1 handles 256–511, and so on. `tl.arange` builds those offsets. `offsets < n` creates a mask for the final partial block. `tl.load` reads the two values and `tl.store` writes their sum.

A block size is a tuning choice, not a promise that 256 is best. We will test other values after establishing a correct baseline.

## 5. Run the repeatable benchmark

The script warms up the operations, measures groups of calls with CUDA events, repeats the measurement, and reports the median time per call. Compilation is triggered before timing. The output buffers are allocated before timing for both implementations.

Do not assume Triton will win: for simple addition, launch overhead can matter a lot.

In [ ]:
!python benchmarks/bench_vector_add.py --size 1048576 --block-size 256 --save

## 6. Record what you observed

After running this notebook, note your actual GPU, library versions, times, and the PyTorch/Triton ratio. A ratio above 1 means PyTorch took more time than Triton in this run; below 1 means Triton took more time. Neither result is universal.

Next, vary one factor at a time (first input size, then block size). Do not change several settings at once, because then it becomes harder to explain which change affected the result.